# Bitcoin Slips Below $80K as Senate Confirms Warsh; $630M Flees Spot ETFs

**Daily Brief — Wednesday, May 13, 2026**

*Generated from `cryptodatascience.com/intelligence`. Every claim below is backed by a verifiable excerpt from a source article — the queries that produce each citation are shown alongside the prose.*

---

A bruising session for crypto markets played out against a heavy political and macro backdrop. The U.S. Senate **confirmed Kevin Warsh as Federal Reserve Chair in a 54–45 vote**, the same day **U.S. spot Bitcoin ETFs posted $630 million in outflows** — their largest daily exit since January. Bitcoin slipped below the **$80,000** mark as April CPI printed at **3.8% year-over-year** and Trump–Xi tensions over Taiwan added to risk-off pressure. In the corporate ledger, **Metaplanet disclosed a $725.6 million Q1 loss** driven by Bitcoin mark-to-market markdowns, **BitGo shares traded ~36% below their offering price**, and both **Consensys and Ledger paused their U.S. IPO plans**.

This brief synthesizes **390 facts from 130 articles across 8 sources**, all keyed to May 13 in `intelligence.facts`. Methodology is at the end.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / 'etl').exists() else Path.cwd().parent
sys.path.insert(0, str(repo_root))

import pandas as pd
from sqlalchemy import text

from etl.common.db.postgres import get_engine

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.width', 220)

engine = get_engine()
BRIEF_DATE = '2026-05-13'

# Quick sanity check on the day's coverage.
pd.read_sql(text('''
    SELECT f.date_referenced AS day,
           COUNT(*) AS facts,
           COUNT(DISTINCT s.source_name) AS sources,
           COUNT(DISTINCT a.article_id) AS articles
      FROM intelligence.facts f
      JOIN intelligence.articles a USING (article_id)
      JOIN intelligence.sources  s USING (source_id)
     WHERE f.date_referenced = :d
     GROUP BY f.date_referenced
'''), engine, params={'d': BRIEF_DATE})

## By the numbers

The day's quantitative facts, ranked by impact. Each row carries the source outlet and a verbatim excerpt — the same span the LLM extracted at ingest time and that anyone can verify against the original article.

In [ ]:
pd.read_sql(text('''
    SELECT s.source_name,
           f.fact_text,
           f.quantity_value AS qty,
           f.quantity_unit AS unit,
           f.fact_type
      FROM intelligence.facts f
      JOIN intelligence.articles a USING (article_id)
      JOIN intelligence.sources  s USING (source_id)
     WHERE f.date_referenced = :d
       AND f.quantity_value IS NOT NULL
       AND f.fact_type IN ('price_event','holdings','institutional','macro','market','regulatory')
     ORDER BY f.confidence DESC, f.quantity_value DESC NULLS LAST
     LIMIT 20
'''), engine, params={'d': BRIEF_DATE})

## The Federal Reserve gets a new chair — by four votes

The Senate's confirmation of **Kevin Warsh** as Fed Chair was the day's biggest macro headline. The vote split **54–45**; Warsh, now 56, becomes the 11th Fed chair of the modern banking era after a tightening campaign nominated him in January. The confirmation followed his earlier Senate Board-of-Governors confirmation on May 12 (a tighter 51–45 margin). Crypto-policy watchers noted Warsh's prior advisory work with the index manager **Bitwise** and stablecoin project **Basis** — financial-tech ties that make him an unusually digital-asset-literate chair by recent Fed standards.

**Why the corroboration cluster matters here:** four different outlets independently report the 54–45 figure, with embedding cosine distances under 0.15 — that's our threshold for "different outlets, same event." The cluster query below shows how many sources confirm each angle.

In [ ]:
# Every fact about Warsh on May 13, with verbatim source excerpts.
pd.read_sql(text('''
    SELECT s.source_name,
           f.fact_text,
           fs.excerpt AS source_excerpt
      FROM intelligence.facts f
      JOIN intelligence.articles a USING (article_id)
      JOIN intelligence.sources  s USING (source_id)
      JOIN intelligence.fact_sources fs USING (fact_id)
     WHERE f.date_referenced = :d
       AND (f.fact_text ILIKE %(warsh)s OR fs.excerpt ILIKE %(warsh)s)
     ORDER BY s.source_name, f.confidence DESC
'''), engine, params={'d': BRIEF_DATE, 'warsh': '%Warsh%'})

## Bitcoin under $80K — the macro vise tightens

Bitcoin spent the session sliding through technical support after testing a resistance zone near **$82,000**. By midday it had broken through **$81,000** and threatened to lose the **$80,000** mark, with traders citing three converging pressures:

1. **U.S. April PPI data** that re-ignited inflation fears
2. **Trump–Xi talks** on Taiwan tensions creating geopolitical risk-off flow
3. **Heavy ETF outflows** — institutions disclosed holdings of **more than 513,000 BTC** via spot ETFs, but on May 13 those vehicles bled **$630 million** in net redemptions, the largest single day since January.

Alpha Vantage's pre-computed per-ticker sentiment for BTC on the day skewed slightly bullish (~+0.09 mean) despite the price action — newsrooms covering the dip framed the moves as macro-driven, not crypto-native.

**Numerical drift in the ETF flow figures is worth flagging:** different outlets reported $630M, $635M, $243M, and $635.2M for what appears to be the same flow event. That kind of disagreement is exactly what the warehouse's corroboration layer is designed to surface for human review.

In [ ]:
# Bitcoin price, ETF flow, and macro facts for May 13.
pd.read_sql(text('''
    SELECT s.source_name,
           f.fact_text,
           f.quantity_value AS qty,
           f.quantity_unit AS unit
      FROM intelligence.facts f
      JOIN intelligence.articles a USING (article_id)
      JOIN intelligence.sources  s USING (source_id)
     WHERE f.date_referenced = :d
       AND f.subject IN ('btc', 'macro', 'market')
       AND f.quantity_value IS NOT NULL
     ORDER BY f.quantity_value DESC NULLS LAST
     LIMIT 25
'''), engine, params={'d': BRIEF_DATE})

In [ ]:
# Alpha Vantage's pre-computed sentiment for BTC across articles tagged on May 13.
pd.read_sql(text('''
    SELECT ats.ticker_sentiment_label AS sentiment,
           COUNT(*) AS articles,
           ROUND(AVG(ats.ticker_sentiment_score)::numeric, 3) AS avg_score,
           ROUND(AVG(ats.relevance_score)::numeric, 3) AS avg_relevance
      FROM intelligence.article_ticker_sentiment ats
      JOIN intelligence.articles a USING (article_id)
     WHERE ats.canonical_asset = 'BTC'
       AND a.published_at::date = :d
     GROUP BY ats.ticker_sentiment_label
     ORDER BY articles DESC
'''), engine, params={'d': BRIEF_DATE})

## Inflation snapshot: April CPI at 3.8%

The Bureau of Labor Statistics reported April CPI rose **3.8% year-over-year** before seasonal adjustment — a print that, combined with hotter-than-expected PPI, pushed back the easing-cycle expectation that had been buoying crypto into early May. Two outlets carry the headline figure with near-identical phrasing, an example of a strong corroboration cluster: cosine distance 0.135.

In [ ]:
# CPI / PPI cluster.
pd.read_sql(text('''
    SELECT s.source_name,
           f.fact_text,
           fs.excerpt AS source_excerpt
      FROM intelligence.facts f
      JOIN intelligence.articles a USING (article_id)
      JOIN intelligence.sources  s USING (source_id)
      JOIN intelligence.fact_sources fs USING (fact_id)
     WHERE f.date_referenced = :d
       AND (f.fact_text ILIKE %(cpi)s OR f.fact_text ILIKE %(ppi)s OR fs.excerpt ILIKE %(cpi)s)
     ORDER BY s.source_name
'''), engine, params={'d': BRIEF_DATE, 'cpi': '%CPI%', 'ppi': '%PPI%'})

## Regulation: CLARITY Act primed for Senate Banking markup

Crypto's marquee regulatory bill — the **Digital Asset Market Clarity Act of 2025 (H.R. 3633)** — entered its decisive week. The Senate Banking Committee scheduled a markup hearing for **Thursday, May 14** at 10:30 a.m. ET; more than **100 amendments** were filed in the run-up, mirroring the 137 amendments that derailed a January attempt by the Senate Banking Committee. Coinbase CEO **Brian Armstrong** issued a public appeal from Capitol Hill — *"CLARITY is closer than ever. The bill is strong."* — corroborated by both `cointelegraph` and Alpha Vantage's news feed.

Committee Chairman **Tim Scott** set a target of June–July 2026 for a full Senate floor vote. The bill's structural provisions remain contentious:

- **Section 404** prohibits stablecoin issuers from paying yield on balances when that yield is the functional equivalent of bank interest
- **Section 604** covers developer protections — whether non-custodial software authors can be treated as financial intermediaries
- **Jurisdictional split**: CFTC gets exclusive jurisdiction over spot and cash markets for digital commodities; SEC retains authority over investment-contract assets and primary-market fundraising

In [ ]:
# All CLARITY Act facts on the day, plus the Coinbase corroboration.
pd.read_sql(text('''
    SELECT s.source_name,
           f.fact_text,
           fs.excerpt AS source_excerpt
      FROM intelligence.facts f
      JOIN intelligence.articles a USING (article_id)
      JOIN intelligence.sources  s USING (source_id)
      JOIN intelligence.fact_sources fs USING (fact_id)
     WHERE f.date_referenced = :d
       AND (f.fact_text ILIKE %(clarity)s OR fs.excerpt ILIKE %(clarity)s
            OR f.fact_text ILIKE %(armstrong)s OR fs.excerpt ILIKE %(armstrong)s)
     ORDER BY s.source_name, f.confidence DESC
     LIMIT 20
'''), engine, params={'d': BRIEF_DATE, 'clarity': '%CLARITY%', 'armstrong': '%Armstrong%'})

## Corporate ledger

The day's deal flow and earnings releases tilted defensive:

- **Metaplanet** posted a **$725.6 million Q1 net loss**, driven by mark-to-market markdowns on its Bitcoin treasury holdings. (Note: `cointelegraph` reports $728M — a small rounding drift on the same fact.)
- **BitGo** shares traded **~36% below their offering price** from the January debut at $18/share that raised about **$213 million**. Wider Q1 losses of **$60.7 million** weighed on the stock despite revenue more than doubling YoY to $3.8 billion.
- **Consensys delayed its planned U.S. IPO until at least fall 2026**, citing weak and volatile market conditions — corroborated by both `cryptopolitan` (long-form) and a `coindesk` headline.
- **Ledger paused its U.S. IPO and stock-market listing plans**, also citing market conditions.
- **KDDI Corporation** agreed to acquire a **14.9% stake in Coincheck Group N.V. for approximately $65 million** — the rare ASEAN-region acquisition story in an otherwise risk-off day.

Below: every corporate fact for May 13, ranked by the dollar value involved.

In [ ]:
pd.read_sql(text('''
    SELECT s.source_name,
           f.fact_text,
           f.quantity_value AS qty,
           f.quantity_unit AS unit
      FROM intelligence.facts f
      JOIN intelligence.articles a USING (article_id)
      JOIN intelligence.sources  s USING (source_id)
     WHERE f.date_referenced = :d
       AND f.fact_type = 'institutional'
     ORDER BY f.quantity_value DESC NULLS LAST
     LIMIT 25
'''), engine, params={'d': BRIEF_DATE})

## Methodology

Every claim in this brief is sourced. Three observations about how the warehouse made this article possible at all:

1. **Cross-source corroboration is automated.** The Warsh confirmation, BitGo trading discount, CLARITY Act markup scheduling, and ETF outflow figures all surfaced as cosine-distance-near-zero pairs across different outlets via `intelligence.facts.embedding <=> intelligence.facts.embedding`. Without that pairing, choosing what to lead with would be guesswork.
2. **Numerical drift is preserved as signal, not silenced.** When `the_block` says "$725.6M" and `cointelegraph` says "$728M" for the same Metaplanet Q1 loss, both numbers are kept and surfaced — the brief notes both. The same applies to the ETF outflow figures.
3. **Headlines pull their weight.** CoinDesk's RSS is title-only (no body), but the LLM extracts atomic claims from headlines like *"Consensys has delayed its potential IPO until fall 2026"* and embeds them — so a one-line CoinDesk headline can corroborate a 2,000-word Cryptopolitan article on the same event.

**Coverage stats for May 13:**

In [ ]:
# Coverage by source for May 13.
pd.read_sql(text('''
    SELECT s.source_name,
           s.source_tier,
           COUNT(DISTINCT a.article_id) AS articles,
           COUNT(f.fact_id) AS facts
      FROM intelligence.facts f
      JOIN intelligence.articles a USING (article_id)
      JOIN intelligence.sources  s USING (source_id)
     WHERE f.date_referenced = :d
     GROUP BY s.source_name, s.source_tier
     ORDER BY facts DESC
'''), engine, params={'d': BRIEF_DATE})

In [ ]:
# Cross-article corroboration density for the day — pairs of facts from
# DIFFERENT outlets whose embeddings are within cosine distance 0.15.
# A high count here means the brief's top stories are independently
# verified across the source set.
pd.read_sql(text('''
    SELECT COUNT(*) AS strong_corroboration_pairs
      FROM intelligence.facts f1
      JOIN intelligence.facts f2
        ON f1.fact_id < f2.fact_id
       AND f1.article_id <> f2.article_id
      JOIN intelligence.articles a1 ON a1.article_id = f1.article_id
      JOIN intelligence.articles a2 ON a2.article_id = f2.article_id
      JOIN intelligence.sources  s1 ON s1.source_id = a1.source_id
      JOIN intelligence.sources  s2 ON s2.source_id = a2.source_id
     WHERE f1.date_referenced = :d
       AND f2.date_referenced = :d
       AND s1.source_name <> s2.source_name
       AND f1.embedding <=> f2.embedding < 0.15
'''), engine, params={'d': BRIEF_DATE})

---

*Brief produced from the cryptodatascience intelligence warehouse on 2026-05-14. Facts extracted by `claude-sonnet-4-6@v2`; embeddings via `text-embedding-3-small` (1536-dim, `pgvector` HNSW). All `fact_sources.excerpt` values are verbatim spans from the source articles — anti-hallucination guard rejects any fact whose excerpt doesn't appear in the original text. To dig deeper, point any of the queries above at a different `BRIEF_DATE`.*